# Crime Analysis Dashboard

Dashboard built from the same visualization charts.


In [1]:
import pandas as pd
import panel as pn
import plotly.express as px
import plotly.graph_objects as go

pn.extension("plotly", sizing_mode="stretch_width")


In [2]:
df = pd.read_csv("cleaned_data.csv", low_memory=False)

df["State"] = df["State"].fillna("Unknown")
df["City"] = df["City"].fillna("Unknown")
df["Weapon_Group"] = df["Weapon_Group"].fillna("Unknown")
df["Relationship_Group"] = df["Relationship_Group"].fillna("Unknown")
df["Perpetrator Sex"] = df["Perpetrator Sex"].fillna("Unknown")
df["Incident"] = pd.to_numeric(df["Incident"], errors="coerce").fillna(1)
df["Year"] = df["Year"].astype(int)

df_trend = df[df["Year"] < 2004].copy()
df_trend.head()


,Record ID,Agency Code,Agency Name,Agency Type,City,State,Year,Month,Incident,Crime Type,...,Perpetrator Race,Perpetrator Ethnicity,Relationship,Weapon,Victim Count,Perpetrator Count,Record Source,Weapon_Group,Relationship_Group,Perpetrator_Age_Available
0,1,AK00101,Anchorage,Municipal Police,Anchorage,Alaska,1980,January,1.0,Murder or Manslaughter,...,Native American/Alaska Native,NaN,Acquaintance,Blunt Object,0.0,0.0,FBI,Blunt Object,Known,True
1,2,AK00101,Anchorage,Municipal Police,Anchorage,Alaska,1980,March,1.0,Murder or Manslaughter,...,White,NaN,Acquaintance,Strangulation,0.0,0.0,FBI,Other,Known,True
2,3,AK00101,Anchorage,Municipal Police,Anchorage,Alaska,1980,March,2.0,Murder or Manslaughter,...,NaN,NaN,NaN,NaN,0.0,0.0,FBI,Unknown,Unknown,False
3,4,AK00101,Anchorage,Municipal Police,Anchorage,Alaska,1980,April,1.0,Murder or Manslaughter,...,White,NaN,Acquaintance,Strangulation,0.0,0.0,FBI,Other,Known,True
4,5,AK00101,Anchorage,Municipal Police,Anchorage,Alaska,1980,April,2.0,Murder or Manslaughter,...,NaN,NaN,NaN,NaN,0.0,1.0,FBI,Unknown,Unknown,False


In [3]:
BG = "#0A0F1C"
CARD = "#111827"
GLOW = "#1A1F3A"
TEXT = "#FFFFFF"
SECONDARY_TEXT = "#A0AEC0"
MUTED = "#6B7280"
GRID = "#1A1F3A"
CYAN = "#00D4FF"
PURPLE = "#7B61FF"
PINK = "#FF4D8D"
PURPLE_2 = "#9D4EDD"
CYAN_2 = "#4CC9F0"
HOVER = "#22D3EE"

weapon_color_map = {
    "Firearm": PURPLE,
    "Sharp Object": CYAN,
    "Blunt Object": PINK,
    "Other": PURPLE_2,
    "Unknown": CYAN_2
}

gender_color_map = {
    "Male": CYAN,
    "Female": PINK,
    "Unknown": MUTED
}

category_orders = {
    "Weapon_Group": ["Blunt Object", "Firearm", "Sharp Object", "Other", "Unknown"],
    "Relationship_Group": ["Family", "Known", "Partner", "Sibling", "Stranger", "Unknown"]
}


In [4]:
yearly_incidents = (
    df_trend[["Year", "Month", "City", "Incident"]]
    .drop_duplicates()
    .groupby("Year")
    .size()
    .reset_index(name="Total_Incidents")
)

year_slider = pn.widgets.IntRangeSlider(
    name="Select Year Range",
    start=int(yearly_incidents["Year"].min()),
    end=int(yearly_incidents["Year"].max()),
    value=(int(yearly_incidents["Year"].min()), int(yearly_incidents["Year"].max())),
    step=1,
    sizing_mode="stretch_width"
)

state_slicer = pn.widgets.MultiChoice(
    name="State",
    options=sorted(df_trend["State"].unique().tolist()),
    value=[],
    solid=False,
    sizing_mode="stretch_width"
)

weapon_slicer = pn.widgets.MultiChoice(
    name="Weapon Group",
    options=sorted(df_trend["Weapon_Group"].unique().tolist()),
    value=[],
    solid=False,
    sizing_mode="stretch_width"
)

relationship_slicer = pn.widgets.MultiChoice(
    name="Relationship Group",
    options=sorted(df_trend["Relationship_Group"].unique().tolist()),
    value=[],
    solid=False,
    sizing_mode="stretch_width"
)

gender_slicer = pn.widgets.MultiChoice(
    name="Perpetrator Sex",
    options=sorted(df_trend["Perpetrator Sex"].unique().tolist()),
    value=[],
    solid=False,
    sizing_mode="stretch_width"
)


In [5]:
def apply_filters(year_range, states, weapons, relationships, genders):
    data = df_trend[
        (df_trend["Year"] >= year_range[0]) &
        (df_trend["Year"] <= year_range[1])
    ]
    if states:
        data = data[data["State"].isin(states)]
    if weapons:
        data = data[data["Weapon_Group"].isin(weapons)]
    if relationships:
        data = data[data["Relationship_Group"].isin(relationships)]
    if genders:
        data = data[data["Perpetrator Sex"].isin(genders)]
    return data

filtered_data = pn.bind(apply_filters, year_slider, state_slicer, weapon_slicer, relationship_slicer, gender_slicer)


In [6]:
def style_fig(fig, height=400):
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(color=TEXT, family="Arial"),
        title=dict(font=dict(color=TEXT, size=15), x=0.5),
        height=height,
        autosize=True,
        margin=dict(l=12, r=12, t=34, b=18),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color=SECONDARY_TEXT, size=10))
    )
    fig.update_xaxes(gridcolor=GRID, zerolinecolor=GRID, tickfont=dict(color=SECONDARY_TEXT, size=10), title_font=dict(color=SECONDARY_TEXT, size=11))
    fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID, tickfont=dict(color=SECONDARY_TEXT, size=10), title_font=dict(color=SECONDARY_TEXT, size=11))
    return fig

def chart_card(chart):
    return pn.Column(
        chart,
        styles={
            "background": f"linear-gradient(145deg, {CARD}, {GLOW})",
            "border": "1px solid rgba(123, 97, 255, 0.5)",
            "border-radius": "8px",
            "padding": "6px",
            "box-shadow": "0 0 18px rgba(123, 97, 255, 0.5)",
            "overflow": "hidden"
        },
        sizing_mode="stretch_width"
    )

def insights_card():
    return pn.pane.HTML(
        f"""
        <div class="insight-card">
            <div class="insight-label">Key Insights</div>
            <ul>
                <li>Incidents rise in phases and peak around 1992-1993 before a strong late-1990s decline.</li>
                <li>California, Texas, and New York dominate the geographic concentration of incidents.</li>
                <li>Firearm usage dominates the distribution, showing strong method concentration.</li>
                <li>Male perpetrator involvement remains much higher across weapon categories.</li>
                <li>Unknown relationship records are large, so relationship findings should be read with caution.</li>
            </ul>
        </div>
        """,
        sizing_mode="stretch_width",
        height=430
    )

def treemap_legend():
    return pn.pane.HTML(
        f"""
        <div class="treemap-legend">
            <div class="legend-item"><span class="legend-swatch" style="background:{weapon_color_map['Firearm']}"></span>Firearm</div>
            <div class="legend-item"><span class="legend-swatch" style="background:{weapon_color_map['Sharp Object']}"></span>Sharp Object</div>
            <div class="legend-item"><span class="legend-swatch" style="background:{weapon_color_map['Blunt Object']}"></span>Blunt Object</div>
            <div class="legend-item"><span class="legend-swatch" style="background:{weapon_color_map['Other']}"></span>Other</div>
            <div class="legend-item"><span class="legend-swatch" style="background:{weapon_color_map['Unknown']}"></span>Unknown</div>
        </div>
        """,
        sizing_mode="stretch_width"
    )


In [7]:
def bubble_chart(data):
    bubble_data = (
        data[["Year", "Relationship_Group", "Weapon_Group", "Perpetrator Sex", "Incident"]]
        .drop_duplicates()
        .groupby(["Year", "Relationship_Group", "Weapon_Group", "Perpetrator Sex"])
        .size()
        .reset_index(name="Total_Incidents")
    )

    bubble_data = bubble_data[bubble_data["Relationship_Group"] != "Unknown"]
    bubble_data = bubble_data[bubble_data["Weapon_Group"] != "Unknown"]
    years = sorted(bubble_data["Year"].unique())
    initial_year = years[0]
    df_init = bubble_data[bubble_data["Year"] == initial_year]
    male_init = df_init[df_init["Perpetrator Sex"] == "Male"]
    female_init = df_init[df_init["Perpetrator Sex"] == "Female"]
    size_ref = 2.0 * max(bubble_data["Total_Incidents"]) / (35.0 ** 2)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=male_init["Weapon_Group"],
        y=male_init["Relationship_Group"],
        mode="markers",
        marker=dict(size=male_init["Total_Incidents"], color=gender_color_map["Male"], sizemode="area", sizeref=size_ref, opacity=0.65, line=dict(width=1.5, color="white")),
        name="Male"
    ))
    fig.add_trace(go.Scatter(
        x=female_init["Weapon_Group"],
        y=female_init["Relationship_Group"],
        mode="markers",
        marker=dict(size=female_init["Total_Incidents"], color=gender_color_map["Female"], sizemode="area", sizeref=size_ref, opacity=0.65, line=dict(width=1.5, color="white")),
        name="Female"
    ))
    fig.update_traces(
        hovertemplate="<b>Relationship:</b> %{y}<br><b>Weapon:</b> %{x}<br><b>Incidents:</b> %{marker.size}<extra></extra>"
    )

    frames = []
    for year in years:
        df_year = bubble_data[bubble_data["Year"] == year]
        male = df_year[df_year["Perpetrator Sex"] == "Male"]
        female = df_year[df_year["Perpetrator Sex"] == "Female"]
        frames.append(dict(
            name=str(year),
            data=[
                dict(x=male["Weapon_Group"], y=male["Relationship_Group"], mode="markers", marker=dict(size=male["Total_Incidents"], color=gender_color_map["Male"], sizemode="area", sizeref=size_ref, opacity=0.65, line=dict(width=1.5, color="white")), name="Male"),
                dict(x=female["Weapon_Group"], y=female["Relationship_Group"], mode="markers", marker=dict(size=female["Total_Incidents"], color=gender_color_map["Female"], sizemode="area", sizeref=size_ref, opacity=0.65, line=dict(width=1.5, color="white")), name="Female")
            ],
            layout=dict(title_text=f"Multi-variable Analysis (Year: {year})")
        ))

    fig.frames = frames
    fig.update_layout(
        title=f"Multi-variable Analysis (Year: {initial_year})",
        sliders=[{
            "active": 0,
            "x": 0.1,
            "y": -0.05,
            "len": 0.78,
            "font": {"color": TEXT},
            "steps": [
                {"label": str(year), "method": "animate", "args": [[str(year)], {"mode": "immediate", "frame": {"duration": 0}, "transition": {"duration": 0}}]}
                for year in years
            ]
        }]
    )
    fig.update_xaxes(categoryorder="array", categoryarray=category_orders["Weapon_Group"])
    fig.update_yaxes(categoryorder="array", categoryarray=category_orders["Relationship_Group"], tickfont=dict(size=12))
    return style_fig(fig, height=410)


In [8]:
def trend_plot(data):
    yearly = (
        data[["Year", "Month", "City", "Incident"]]
        .drop_duplicates()
        .groupby("Year")
        .size()
        .reset_index(name="Total_Incidents")
    )
    fig = px.line(yearly, x="Year", y="Total_Incidents", title="Incident Trend Over Time", markers=True)
    fig.update_traces(line=dict(width=3, color=CYAN), marker=dict(size=7, color=CYAN))
    fig.update_layout(showlegend=False)
    return style_fig(fig, height=240)

def state_bar_chart(data):
    state_data = (
        data[["State", "Year", "Month", "City", "Incident"]]
        .drop_duplicates()
        .groupby("State")
        .size()
        .reset_index(name="Total_Incidents")
        .sort_values(by="Total_Incidents", ascending=False)
        .head(10)
    )
    fig = px.bar(state_data, x="Total_Incidents", y="State", orientation="h", title="Top 10 States by Incidents", color_discrete_sequence=[PURPLE])
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
    return style_fig(fig, height=240)

def weapon_donut_chart(data):
    weapon_data = (
        data[["Weapon_Group", "Year", "Month", "City", "Incident"]]
        .drop_duplicates()
        .groupby("Weapon_Group")
        .size()
        .reset_index(name="Total_Incidents")
        .sort_values(by="Total_Incidents", ascending=False)
    )
    top_weapons = weapon_data[weapon_data["Weapon_Group"] != "Unknown"].head(6)
    fig = px.pie(top_weapons, names="Weapon_Group", values="Total_Incidents", hole=0.5, color="Weapon_Group", color_discrete_map=weapon_color_map, title="Weapon Distribution")
    fig.update_traces(textinfo="percent", textposition="inside", textfont=dict(color="white"))
    fig.update_layout(legend=dict(orientation="v", x=1, y=0.5))
    return style_fig(fig, height=240)

def relationship_treemap_chart(data):
    fig = px.treemap(
        data,
        path=["Relationship_Group", "Weapon_Group"],
        values="Incident",
        color="Incident",
        color_continuous_scale=[CYAN, PURPLE],
        title="Hierarchical Analysis of Victim-Perpetrator Relationships and Weapon Usage"
    )
    fig.update_traces(hovertemplate="<b>%{label}</b><br>Incidents: %{value}<extra></extra>")
    fig.update_coloraxes(colorbar_title="Incident")
    return style_fig(fig, height=260)

def heatmap_chart(data):
    heatmap_data = (
        data[["Perpetrator Sex", "Weapon_Group", "Incident"]]
        .drop_duplicates()
        .groupby(["Perpetrator Sex", "Weapon_Group"])
        .size()
        .reset_index(name="Total_Incidents")
    )
    heatmap_pivot = heatmap_data.pivot(index="Perpetrator Sex", columns="Weapon_Group", values="Total_Incidents").fillna(0)
    heatmap_pivot = heatmap_pivot.drop(columns=["Unknown"], errors="ignore")
    fig = px.imshow(heatmap_pivot, text_auto=True, color_continuous_scale=[BG, CYAN, PURPLE], aspect="auto", title="Gender-wise Distribution of Weapon Usage")
    fig.update_traces(hovertemplate="<b>Gender:</b> %{y}<br><b>Weapon:</b> %{x}<br><b>Incidents:</b> %{z}<extra></extra>")
    fig.update_layout(hovermode="closest")
    return style_fig(fig, height=260)


In [9]:
bubble_panel = pn.bind(bubble_chart, filtered_data)
trend_panel = pn.bind(trend_plot, filtered_data)
state_panel = pn.bind(state_bar_chart, filtered_data)
weapon_panel = pn.bind(weapon_donut_chart, filtered_data)
relationship_panel = pn.bind(relationship_treemap_chart, filtered_data)
heatmap_panel = pn.bind(heatmap_chart, filtered_data)


In [10]:
pn.config.raw_css.append(f"""
body {{
    background: radial-gradient(circle at top left, {GLOW} 0%, {BG} 36%, {BG} 100%);
}}
.bk-root {{
    background: transparent !important;
    color: {TEXT};
}}
.sidebar {{
    background: linear-gradient(180deg, {CARD}, {GLOW}) !important;
    border-right: 1px solid rgba(123, 97, 255, 0.28);
}}
.dashboard-title {{
    background: linear-gradient(135deg, rgba(123, 97, 255, 0.18), rgba(0, 212, 255, 0.06));
    border: 1px solid rgba(123, 97, 255, 0.5);
    border-radius: 8px;
    padding: 14px 18px;
    box-shadow: 0 0 22px rgba(123, 97, 255, 0.5);
    text-align: center;
}}
.dashboard-title h1 {{
    color: {TEXT};
    font-size: 38px;
    line-height: 1.05;
    margin: 0;
    font-weight: 800;
}}
.dashboard-title .accent {{
    color: {PURPLE};
    text-shadow: 0 0 18px rgba(123, 97, 255, 0.5);
}}
.dashboard-title .cyan {{
    color: {CYAN};
    text-shadow: 0 0 18px rgba(0, 212, 255, 0.35);
}}
.dashboard-title p {{
    color: {SECONDARY_TEXT};
    margin-top: 8px;
    font-size: 14px;
}}
.insight-card {{
    height: 100%;
    background: linear-gradient(145deg, {CARD}, {GLOW});
    border: 1px solid rgba(123, 97, 255, 0.5);
    border-radius: 8px;
    padding: 14px 16px;
    box-shadow: 0 0 18px rgba(123, 97, 255, 0.5);
    color: {SECONDARY_TEXT};
    overflow: hidden;
}}
.insight-label {{
    color: {TEXT};
    font-size: 18px;
    font-weight: 700;
    margin-bottom: 8px;
}}
.insight-card ul {{ margin: 0; padding-left: 18px; }}
.insight-card li {{ margin-bottom: 8px; font-size: 13px; line-height: 1.35; }}
.bk-input, .bk-input-group input, .bk-input-group select {{
    background-color: #151B2E !important;
    color: {TEXT} !important;
    border-color: rgba(123, 97, 255, 0.5) !important;
}}
.bk-input:focus {{ border-color: {HOVER} !important; }}
.bk-btn:hover, .bk-input:hover {{ border-color: {HOVER} !important; }}
""")


In [11]:
sidebar = pn.Column(
    pn.pane.Markdown("## Filters"),
    year_slider,
    state_slicer,
    weapon_slicer,
    relationship_slicer,
    gender_slicer,
    pn.Spacer(height=28),
    insights_card(),
    pn.Spacer(height=20),
    pn.pane.Markdown("""### Very Important
The multi-variable bubble analysis is placed first in the dashboard and also has its own animation slider."""),
    styles={"background": CARD, "padding": "8px"},
    sizing_mode="stretch_width"
)

template = pn.template.FastGridTemplate(
    title="Crime Analysis Dashboard",
    theme="dark",
    accent_base_color=PURPLE,
    header_background=BG,
    sidebar=[sidebar],
    row_height=84,
    prevent_collision=True,
    save_layout=True
)

template.main[0:2, 0:12] = pn.pane.HTML(
    """
    <div class="dashboard-title">
        <h1><span class="accent">Homicides Pattern Analysis</span> <span class="cyan">Across The World</span> From 1980 - 2003</h1>
        <p>Interactive view of multi-variable patterns, time trend, geography, weapon usage, relationship hierarchy, and gender-wise weapon distribution.</p>
    </div>
    """
)

template.main[2:8, 0:12] = chart_card(bubble_panel)
template.main[8:12, 0:6] = chart_card(trend_panel)
template.main[8:12, 6:12] = chart_card(state_panel)
template.main[12:16, 0:4] = chart_card(weapon_panel)
template.main[12:16, 4:8] = chart_card(relationship_panel)
template.main[12:16, 8:12] = chart_card(heatmap_panel)

template.servable()
template


FastGridTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [nav-1381751856432] Column(sizing_mode='stretch_width', styles={'background': '#111827', ...})
        [0] Markdown(str, sizing_mode='stretch_width')
        [1] IntRangeSlider(end=2003, name='Select Year Range', sizing_mode='stretch_width', start=1980, value=(1980, 2003), value_end=2003, value_start=1980)
        [2] MultiChoice(name='State', options=['Alabama', 'Alaska', ...], sizing_mode='stretch_width', solid=False)
        [3] MultiChoice(name='Weapon Group', options=['Blunt Object', ...], sizing_mode='stretch_width', solid=False)
        [4] MultiChoice(name='Relationship Group', options=['Family', 'Known', ...], sizing_mode='stretch_width', solid=False)
        [5] MultiChoice(name='Perpetrator Sex', options=['Female', 'Male', ...], sizing_mode='stretch_width', solid=False)
        [6] Spacer(height=28, sizing_mode='stretch_width')
        [7] HTML(str, height=430, sizing_mode='stretch_width')
        [8] Spacer(height=20, sizing_mode='stretch_width')
        [9] Markdown(str, sizing_mode='stretch_width')
    [main-1381302328224] HTML(str, sizing_mode='stretch_width')
    [main-1381751890448] Column(sizing_mode='stretch_width', styles={'background': 'linear-gra...})
        [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-1381384271296] Column(sizing_mode='stretch_width', styles={'background': 'linear-gra...})
        [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-1381401817392] Column(sizing_mode='stretch_width', styles={'background': 'linear-gra...})
        [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-1381402925904] Column(sizing_mode='stretch_width', styles={'background': 'linear-gra...})
        [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-1381405239216] Column(sizing_mode='stretch_width', styles={'background': 'linear-gra...})
        [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-1381406466000] Column(sizing_mode='stretch_width', styles={'background': 'linear-gra...})
        [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')

In [12]:
template.show()


Launching server at http://localhost:63628


2026-04-23 11:32:44,455 ERROR: panel.reactive - Callback failed for object named 'Select Year Range' changing properties {'value': (1980, 2003), 'value_throttled': (1980, 2003)} 
Traceback (most recent call last):
  File "C:\Users\anany\AppData\Local\Programs\Python\Python314\Lib\site-packages\panel\reactive.py", line 388, in _process_events
    self.param.update(**self_params)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\anany\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 2318, in update
    restore = dict(self_._update(arg, **kwargs))
                   ~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\anany\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 2351, in _update
    self_._batch_call_watchers()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\anany\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 2545, in _batch_call_watchers
    self_._execute_watc